# 03 · Cleaning and off-the-shelf momentum features

**Unit of analysis:** one row per *market × decision time*.
- 15-minute markets: decide **10 min and 5 min** before close.
- Hourly ladder: decide **30 min and 15 min** before close.

Each row carries the quote at that instant, the quotes 60 s and 180 s earlier, the strike, and the
settlement label.

**Rules learned the hard way (each one once produced a fake edge):**
1. Use the quote at a single instant, never an average over a window.
2. Use only collector-validated, uncrossed quotes at most 5 s old.
3. Confirm every quote against the exchange's own candle.
4. Use only BTC bars that had closed by the decision time.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tabtrader import plots
plots.style()
pd.set_option("display.width", 140, "display.max_columns", 30)

In [2]:
from tabtrader import eda
from tabtrader.data import load_market, MARKETS
for k in ["15m", "1h"]:
    print(MARKETS[k].title)
    for step, n in eda.cleaning_funnel(k):
        print(f"   {n:>7,}  {step}")

KXBTC15M · 15-minute up/down


    29,110  settled markets × decision times
    23,651  valid two-sided quote ≤ 5 s old
    17,736  inside the exchange candle's range
    15,623  within 2¢ of the exchange close
    15,623  BTC bar present
KXBTCD · hourly strike ladder


    26,056  settled markets × decision times
    17,090  valid two-sided quote ≤ 5 s old
    13,471  inside the exchange candle's range
    12,141  within 2¢ of the exchange close
    12,141  BTC bar present
     7,227  mid between 10¢ and 90¢


### Reproducing the published table from the raw week

In [3]:
raw = ROOT / "data" / "raw_week"
q = pd.read_parquet(raw / "kxbtc15m_quotes_1s.parquet")
btc_pub = pd.read_parquet(ROOT / "data" / "btc_1m.parquet")
# 1-minute BTC bars from the 1-second tape: the close is the last price in the minute.
sec = q.groupby("sec_end").btc.last().dropna()
rebuilt = sec[sec.index.second == 0]
rebuilt.index.name = "bar_end"
cmp = btc_pub.set_index("bar_end").close.reindex(rebuilt.index)
print(f"BTC 1-minute closes rebuilt from raw seconds: {np.isclose(cmp, rebuilt).mean():.2%} identical "
      f"({len(rebuilt):,} minutes)")

BTC 1-minute closes rebuilt from raw seconds: 100.00% identical (9,578 minutes)


In [4]:
snap = pd.read_parquet(ROOT / "data" / "snapshots" / "kxbtc15m.parquet")
wk = snap[(snap.decision_ts >= "2026-09-26") & (snap.decision_ts < "2026-10-03") & snap.bid.notna()]
qq = q[q.quote_valid == 1].rename(columns={"sec_end": "decision_ts"})
m = wk.merge(qq[["market", "decision_ts", "bid", "ask"]], on=["market", "decision_ts"], suffixes=("", "_raw"))
same = (m.bid == m.bid_raw) & (m.ask == m.ask_raw)
print(f"decision quotes rebuilt from the 1-second raw tape: {same.mean():.1%} identical "
      f"of {len(m):,} (the rest moved in the final tick before the decision instant)")

decision quotes rebuilt from the 1-second raw tape: 95.6% identical of 1,326 (the rest moved in the final tick before the decision instant)


### The features: about 20 standard indicators, all at textbook settings

| group | features |
|---|---|
| contract | mid price (as probability), its logit, spread, minutes to close, change in mid over the last 60 s and 180 s |
| moneyness | log(BTC / strike); the same in volatility units, z = log(S/K)/(σ√τ); Φ(z), a textbook digital price |
| BTC momentum (1-min bars) | returns over 1/3/5/15/60 min, MACD(12,26,9) and histogram, RSI-14, Bollinger %B(20,2), 10-bar OLS slope and R², position in the 60-min range |
| volatility | realised volatility over 15 and 60 min |
| calendar | hour of day (sine, cosine) |

None of them is tuned, and none of them is the feature set my live strategy uses.

In [5]:
from tabtrader.features import FEATURES, FEATURE_GROUPS
df = load_market("15m")
df[FEATURES].describe().T[["mean", "std", "min", "max"]].round(3)

,mean,std,min,max
p_mid,0.506,0.279,0.002,0.996
logit_mid,0.045,1.641,-5.293,5.293
spread_c,1.293,1.006,0.100,6.000
tau_min,7.831,2.478,5.000,10.000
d_mid_60s,0.152,11.774,-71.650,70.000
d_mid_180s,0.184,18.978,-72.250,81.400
log_moneyness_bps,-0.155,12.732,-179.148,152.527
z_moneyness,-0.019,1.109,-7.159,6.864
phi_z,0.494,0.276,0.000,1.000
ret_1m_bps,0.086,4.533,-38.160,47.837


### How much do these features know beyond the price? (training period only, before 3 August)

In [6]:
from sklearn.metrics import roc_auc_score
tr = df[df.decision_ts < "2026-08-03"]
resid = tr.label - tr.p_mid
info = pd.DataFrame({
    "AUC vs label": {f: roc_auc_score(tr.label, tr[f].fillna(tr[f].median())) for f in FEATURES},
    "corr with (label − price)": {f: np.corrcoef(tr[f].fillna(tr[f].median()), resid)[0, 1] for f in FEATURES},
}).round(3)
info.sort_values("corr with (label − price)", key=abs, ascending=False)

,AUC vs label,corr with (label − price)
phi_z,0.797,0.035
p_mid,0.803,0.031
z_moneyness,0.797,0.030
logit_mid,0.803,0.027
log_moneyness_bps,0.784,0.024
spread_c,0.512,0.021
bb_pctb_20,0.727,0.017
hour_cos,0.498,-0.015
ret_5m_bps,0.711,0.013
ret_60m_bps,0.544,-0.013


Alone, almost every feature looks predictive (AUC well above 0.5), because almost every feature also
tracks the price. Measured against **what the price misses**, each one is a weak, noisy signal.
Combining many weak signals without overfitting a few thousand rows is exactly the problem a
pretrained tabular prior is built for. Notebook 04 tests whether it succeeds.